# 🦺 PPE Detection — GPU Training (YOLO11)

Trains an improved PPE / **helmet (Hardhat)** detector on the construction-site-safety dataset (10 classes).

**Goals**
- Higher helmet accuracy across varied lighting / angles / scale (augmentation below).
- Fewer *hair-detected-as-helmet* false positives — driven by (a) a larger backbone, (b) the dataset's `NO-Hardhat` (bare-head) hard negatives, and (c) the class-aware confidence threshold used at inference in the app.

**How to run**
1. Runtime → Change runtime type → **GPU** (Colab) or enable the **GPU accelerator** (Kaggle).
2. Run the cells top to bottom.
3. Pick a dataset source in the **Dataset setup** cell.
4. Download `best.pt` at the end and drop it next to `app.py` (or set `PPE_MODEL_PATH`).


## 1. Install & check GPU


In [ ]:
!pip -q install "ultralytics>=8.3.0"
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Enable a GPU runtime before training.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Dataset setup

Choose **one** source by setting `DATASET_SOURCE` below.

- `kaggle`  — add the *Construction Site Safety Image Dataset (Roboflow)* to your Kaggle notebook; it mounts at `/kaggle/input/...`.
- `roboflow` — download via the Roboflow API (paste your API key).
- `upload`  — you uploaded/extracted the `css-data` folder yourself; set `CUSTOM_ROOT` to it.

Each split folder must contain `images/` and `labels/` (YOLO format).


In [ ]:
import os, glob, yaml

DATASET_SOURCE = 'kaggle'   # 'kaggle' | 'roboflow' | 'upload'
CUSTOM_ROOT    = '/content/css-data'   # used only when DATASET_SOURCE == 'upload'
ROBOFLOW_API_KEY = ''                  # used only when DATASET_SOURCE == 'roboflow'

if DATASET_SOURCE == 'kaggle':
    # The Roboflow construction-site-safety dataset mounts here on Kaggle.
    root = '/kaggle/input/construction-site-safety-image-dataset-roboflow'
elif DATASET_SOURCE == 'roboflow':
    !pip -q install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    # Update workspace/project/version to your Roboflow project if different.
    project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
    dataset = project.version(30).download('yolov8')
    root = dataset.location
else:  # upload
    root = CUSTOM_ROOT

print('Dataset root:', root)
assert os.path.isdir(root), f'Dataset root not found: {root}'

### Build `data.yaml`
We detect the train/val/test split folders under the dataset root and write a fresh `data.yaml` with the 10 PPE classes.


In [ ]:
def find_split(root, names):
    for n in names:
        for cand in (os.path.join(root, n, 'images'), os.path.join(root, n)):
            if os.path.isdir(cand) and glob.glob(os.path.join(cand, '*')):
                return cand
    return None

train_dir = find_split(root, ['train'])
val_dir   = find_split(root, ['valid', 'val'])
test_dir  = find_split(root, ['test'])
assert train_dir and val_dir, f'Could not locate train/val image folders under {root}'

CLASS_NAMES = ['Hardhat','Mask','NO-Hardhat','NO-Mask','NO-Safety Vest','Person','Safety Cone','Safety Vest','machinery','vehicle']
data = {
    'path': root,
    'train': os.path.relpath(train_dir, root),
    'val': os.path.relpath(val_dir, root),
    'nc': len(CLASS_NAMES),
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}
if test_dir:
    data['test'] = os.path.relpath(test_dir, root)

with open('data.yaml', 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)
print(open('data.yaml').read())
print('train images:', len(glob.glob(os.path.join(train_dir, '*'))))
print('val images:  ', len(glob.glob(os.path.join(val_dir, '*'))))

## 3. Train

Defaults: **YOLO11s**, 120 epochs, 640px. For maximum accuracy and the best hair-vs-helmet separation, try `MODEL = 'yolo11m.pt'` if your GPU has enough memory.

The augmentation is tuned for **various conditions**: `hsv_v` (lighting), `scale` (near/far helmets), `degrees`/`perspective` (camera angles). `cls=0.7` slightly sharpens class separation (Hardhat vs NO-Hardhat vs hair). These mirror the repo's `train.py`.


In [ ]:
from ultralytics import YOLO

MODEL  = 'yolo11s.pt'   # try 'yolo11m.pt' for more accuracy if VRAM allows
EPOCHS = 120
IMGSZ  = 640
NAME   = 'ppe_yolo11s'

model = YOLO(MODEL)
results = model.train(
    data='data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=-1, device=0,
    project='runs/ppe', name=NAME,
    optimizer='auto', cos_lr=True, patience=30, close_mosaic=15, seed=0,
    box=7.5, cls=0.7, dfl=1.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    degrees=10.0, translate=0.1, scale=0.5, shear=2.0, perspective=0.0005,
    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.1,
    plots=True, val=True,
)
save_dir = str(model.trainer.save_dir)
print('Saved to:', save_dir)

## 4. Validate — per-class metrics (Hardhat highlighted)


In [ ]:
best = os.path.join(save_dir, 'weights', 'best.pt')
m = YOLO(best)
metrics = m.val(data='data.yaml', split='val', imgsz=IMGSZ)
names = m.names
print('\n=== Per-class mAP50 / mAP50-95 ===')
for i, c in enumerate(metrics.box.ap_class_index):
    name = names[int(c)]
    mark = '   <-- HELMET' if name == 'Hardhat' else ''
    print(f'{name:>16}: mAP50={metrics.box.ap50[i]:.3f}  mAP50-95={metrics.box.ap[i]:.3f}{mark}')
print(f'\nOverall mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}')

### Training curves & confusion matrix


In [ ]:
from IPython.display import Image, display
for p in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    fp = os.path.join(save_dir, p)
    if os.path.exists(fp):
        print(p); display(Image(filename=fp))

## 5. Sanity check + the hair-vs-helmet threshold

The app applies a **class-aware confidence threshold** (Hardhat/Helmet ≥ 0.55) so low-confidence helmet guesses on hair are dropped. Validate the idea here: run on a few val images and look at the Hardhat confidences.


In [ ]:
import random
val_imgs = glob.glob(os.path.join(val_dir, '*'))
random.seed(0); sample = random.sample(val_imgs, min(6, len(val_imgs)))
preds = m.predict(sample, conf=0.25, iou=0.5, verbose=False)
for f, r in zip(sample, preds):
    dets = [(names[int(c)], round(float(p),3)) for c,p in zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy())]
    print(os.path.basename(f), '->', dets)
    display(Image(data=__import__('cv2').imencode('.jpg', r.plot())[1].tobytes()))

## 6. Download the trained model


In [ ]:
best = os.path.join(save_dir, 'weights', 'best.pt')
print('best.pt at:', best)
try:
    from google.colab import files   # Colab download
    files.download(best)
except Exception:
    # On Kaggle, copy to /kaggle/working so it appears in the Output tab.
    import shutil
    shutil.copy(best, '/kaggle/working/best.pt')
    print('Copied to /kaggle/working/best.pt (see the Output tab).')

---
**After downloading:** put `best.pt` next to `app.py` (replacing the old one) or set `PPE_MODEL_PATH`. 
The Streamlit app auto-detects whether the helmet class is named `Helmet` or `Hardhat`, so no code changes are needed.
